In [1]:
import json
import pandas as pd

from cat_whim.config import BIDS_DATA_DIR

2024-09-12 17:22:35.799 | INFO     | cat_whim.config:<module>:11 - PROJ_ROOT path is: /home/leoner/Projects/cat_whim


In [2]:
def get_dicom_params(BIDS_DATA_DIR):
    list_subj = []
    list_ses = []
    list_seq = []
    list_te = []
    list_tr = []
    list_ti = []
    list_flip = []
    list_field_strength = []
    list_model = []

    for subj_folder in BIDS_DATA_DIR.iterdir():
        if subj_folder.is_dir() and subj_folder.name.startswith("sub-"):
            for ses_folder in subj_folder.iterdir():
                if ses_folder.is_dir() and ses_folder.name.startswith("ses-"):
                    anat_folder = ses_folder / "anat"
                    for json_file in anat_folder.iterdir():
                        if json_file.is_file() and json_file.name.endswith(".json"):
                            
                            
                            with open(json_file) as f:

                                dicom = json.load(f)

                                subj = subj_folder.name
                                ses = ses_folder.name.split("-")[1]
                                seq = f.name.split("_")[-1].split(".")[0]
                                te = dicom["EchoTime"]
                                tr = dicom["RepetitionTime"]
                                flip = dicom["FlipAngle"]
                                field_strength = dicom["MagneticFieldStrength"]
                                model = dicom["ManufacturersModelName"]

                                try:
                                    institution = dicom["InstitutionName"]
                                except:
                                    institution = ""

                                if "flair" in dicom["SeriesDescription"].lower():
                                    ti = dicom["InversionTime"]
                                else:
                                    ti = None

                                list_subj.append(subj)
                                list_ses.append(ses)
                                list_seq.append(seq)
                                list_te.append(te)
                                list_tr.append(tr)
                                list_ti.append(ti)
                                list_flip.append(flip)
                                list_field_strength.append(field_strength)
                                list_model.append(model + "_" + institution)
                                

    df_dicom = pd.DataFrame(
        {
            "PTID": list_subj,
            "session": list_ses,
            "sequence": list_seq,
            "TE": list_te,
            "TR": list_tr,
            "TI": list_ti,
            "FlipAngle": list_flip,
            "FieldStrength": list_field_strength,
            "Model": list_model,
        }
    )

    return df_dicom

df_dicom = get_dicom_params(BIDS_DATA_DIR)

df_dicom_t1 = df_dicom[df_dicom["sequence"] == "T1w"].copy()
df_dicom_flair = df_dicom[df_dicom["sequence"] == "FLAIR"].copy()

print("T1")
print(f"TE = {df_dicom_t1['TE'].min()}-{df_dicom_t1['TE'].max()}")
print(f"TR = {df_dicom_t1['TR'].min()}-{df_dicom_t1['TR'].max()}")

print("FLAIR")
print(f"TE = {df_dicom_flair['TE'].min()}-{df_dicom_flair['TE'].max()}")
print(f"TR = {df_dicom_flair['TR'].min()}-{df_dicom_flair['TR'].max()}")
print(f"TI = {df_dicom_flair['TI'].min()}-{df_dicom_flair['TI'].max()}")

T1
TE = 0.002912-0.003208
TR = 0.006503-2.3
FLAIR
TE = 0.115489-0.445
TR = 4.8-4.802
TI = 1.348-1.8
